In [ ]:
from datasets import load_dataset
import pandas as pd

# 直接讀進 pandas，不用手動下載
ds = load_dataset("c01dsnap/CIC-IDS2017",
                  data_files="Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
                  split="train")
df = ds.to_pandas()

In [ ]:
print(df.shape)
print(df.columns.tolist())
print(df.head(5))
print(df[" Label"].value_counts())

In [ ]:
df

CSV 原始資料
    ↓
Analyst LLM  →  輸出 JSON（摘要 + 時間線 + confidence）
    ↓
Hunter LLM   →  輸出 JSON（ATT&CK + 威脅等級 + confidence）
    ↓
Auditor LLM  →  輸出 JSON（漏洞審查 + revised 威脅等級 + revised confidence）
    ↓
[Debate Round]  →  三個 agent 看彼此輸出後各自修正一次
    ↓
Judge Agent  →  整合三份 JSON，輸出最終結論

In [ ]:
print(df.columns.tolist())

In [ ]:
import pandas as pd

# 先清空格
df.columns = df.columns.str.strip()

# ── Analyst 視角：網路行為特徵（最原始的觀察層）
analyst_cols = [
    "Destination Port", "Flow Duration",
    "Total Fwd Packets", "Total Backward Packets",
    "Flow Bytes/s", "Flow Packets/s",
    "FIN Flag Count", "SYN Flag Count", "RST Flag Count",
    "ACK Flag Count", "PSH Flag Count",
    "Label"
]
df_analyst = df[analyst_cols].copy()

# ── Hunter 視角：封包統計特徵（更高層次的模式）
hunter_cols = [
    "Flow Duration",
    "Fwd Packet Length Max", "Fwd Packet Length Min", "Fwd Packet Length Mean",
    "Bwd Packet Length Max", "Bwd Packet Length Mean",
    "Packet Length Mean", "Packet Length Std", "Packet Length Variance",
    "Average Packet Size",
    "Down/Up Ratio",
    "Init_Win_bytes_forward", "Init_Win_bytes_backward",
    "Label"
]
df_hunter = df[hunter_cols].copy()

# ── 確認
print("Analyst view:", df_analyst.shape)
print(df_analyst.head(3))

print("\nHunter view:", df_hunter.shape)
print(df_hunter.head(3))

print("\nLabel distribution:")
print(df["Label"].value_counts())

In [ ]:
df_hunter

In [ ]:
df_analyst

## llama factory register dataset

In [ ]:
# ── 1. 下載 CICIDS2017 ────────────────────────────────────────────────
from datasets import load_dataset
import pandas as pd
import json, os

print("Downloading CICIDS2017...")
ds = load_dataset(
    "c01dsnap/CIC-IDS2017",
    data_files="Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    split="train"
)
df = ds.to_pandas()
df.columns = df.columns.str.strip()
print(f"Downloaded: {df.shape}, Labels: {df['Label'].value_counts().to_dict()}")

# ── 2. 轉成 Alpaca JSON（Analyst 視角）───────────────────────────────
analyst_cols = [
    "Destination Port", "Flow Duration",
    "Total Fwd Packets", "Total Backward Packets",
    "FIN Flag Count", "SYN Flag Count", "ACK Flag Count", "PSH Flag Count"
]

# 讀取你的完整 df（改成你實際的 df 來源）
# df = pd.read_csv("/你的資料路徑.csv")

analyst_cols = [col for col in df.columns if col != "Label"]

records = []
for _, row in df.iterrows():  # ← 直接拿全部，不 sample
    input_text = ", ".join([f"{col}: {row[col]}" for col in analyst_cols])
    records.append({
        "instruction": "You are a cybersecurity analyst. Classify the following network flow.",
        "input": input_text,
        "output": str(row["Label"])
    })

# 存檔
with open("/LLaMA-Factory/data/cicids_analyst.json", "w") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print("完成！共", len(records), "筆")

# ── 3. 存到 LLaMA Factory 的 data/ 資料夾 ──────────────────────────
os.makedirs("/root/LLaMA-Factory/data", exist_ok=True)
with open("/root/LLaMA-Factory/data/cicids_analyst.json", "w") as f:
    json.dump(records, f, indent=2)
print(f"Saved {len(records)} records to /root/LLaMA-Factory/data/cicids_analyst.json")



In [ ]:
import json

with open("/LLaMA-Factory/data/dataset_info.json", "r") as f:
    info = json.load(f)

# 印出所有已登記的 dataset 名稱
print("目前登記的 datasets：")
for key in info.keys():
    print(" -", key)

In [ ]:
from datasets import load_dataset
import pandas as pd
import json, os

print("Downloading CICIDS2017...")
ds = load_dataset(
    "c01dsnap/CIC-IDS2017",
    data_files="Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    split="train"
)
df = ds.to_pandas()
df.columns = df.columns.str.strip()
print(f"Downloaded: {df.shape}, Labels: {df['Label'].value_counts().to_dict()}")

# ── 定義 label 的完整說明 ──────────────────────────────────────────
label_description = {
    "BENIGN": "This network flow is benign. No malicious activity detected.",
    "Web Attack – Brute Force": "This network flow is a Web Attack – Brute Force. The attacker is repeatedly attempting login credentials.",
    "Web Attack – XSS": "This network flow is a Web Attack – XSS (Cross-Site Scripting). Malicious scripts are being injected into web requests.",
    "Web Attack – Sql Injection": "This network flow is a Web Attack – SQL Injection. Malicious SQL commands are embedded in the request to manipulate the database.",
}

analyst_cols = [col for col in df.columns if col != "Label"]

# ── 平衡取樣（每個 label 最多 1000 筆）──────────────────────────────
df_balanced = df.groupby("Label", group_keys=False).apply(
    lambda x: x.sample(min(len(x), 1000), random_state=42)
)
print(f"平衡後：{df_balanced['Label'].value_counts().to_dict()}")

records = []
for _, row in df_balanced.iterrows():
    label = str(row["Label"]).strip()
    input_text = ", ".join([f"{col}: {row[col]}" for col in analyst_cols])
    # output 改成完整分析句
    output_text = label_description.get(label, f"This network flow is classified as {label}.")
    records.append({
        "instruction": "You are a cybersecurity analyst. Analyze the following network flow and classify it.",
        "input": input_text,
        "output": output_text
    })

# ── 存檔 ────────────────────────────────────────────────────────────
os.makedirs("/root/LLaMA-Factory/data", exist_ok=True)
save_path = "/root/LLaMA-Factory/data/cicids_analyst.json"
with open(save_path, "w") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print(f"\n✅ 完成！共 {len(records)} 筆")
print(f"儲存至：{save_path}")
print(f"\n第一筆 output 範例：\n{records[0]['output']}")